# 第 7 章の講義と実習　比較の可視化

講義では，平均の棒だけでは見えない個体の測定値を，分布と散布図で読みます。後半の約 20 分は翼の長さを使い，知りたい違いに合う図を自分で選びます。

## 準備

Adelie，Chinstrap，Gentoo では，翼の長さが集まる範囲はどう違うでしょうか。それとも，翼が長い個体ほど体重も重いでしょうか。本文と同じ記録について一方の問いを選び，何を一本の棒または一点で表すかを決めて図を作ります。

Colab では「ドライブにコピーを保存」で自分用のノートを作り，上から準備セルを実行します。CSV は公開 URL から読み込むため，アップロードは不要です。ローカルでは，実習 ZIP を展開してノートブックと CSV を同じフォルダーに置くと，手元のファイルを使えます。

準備セルを実行し，3 種類のペンギンで計 342 件の測定を使うことを確かめます。本文と同じく，四つの測定値がそろう行を使います。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

file_name = "penguins.csv"
csv_path = Path("data") / file_name
if not csv_path.is_file():
    csv_path = Path("../data") / file_name
if not csv_path.is_file():
    csv_path = Path(file_name)
if not csv_path.is_file():
    csv_path = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_name

penguins = pd.read_csv(csv_path)
has_measurements = (
    penguins["body_mass_g"].notna()
    & penguins["bill_length_mm"].notna()
    & penguins["bill_depth_mm"].notna()
    & penguins["flipper_length_mm"].notna()
)
measured = penguins.loc[has_measurements]
measured.groupby("species").size()

## 講義中に実行するセル

見出しの順に実行します。セルを動かす前に，比べる行・値・図を確認します。

### 1. 図に入れる測定記録を確かめる

準備セルは，体重・くちばしの長さと厚み・翼の長さがそろう行だけを `measured` に入れました。先頭の行を読み，体重とくちばしの寸法が同じ一羽の測定値として並ぶことを確かめます。後の散布図でも，横と縦には同じ行の値を渡します。

In [ ]:
print("Rows in CSV:", len(penguins), "Rows used:", len(measured))
measured[["species", "body_mass_g", "bill_length_mm", "bill_depth_mm"]].head(3)

### 2. ペンギンの種類ごとに平均体重を比べる

Adelie と Chinstrap の平均体重はどれほど近いでしょうか。横軸はペンギンの種類，縦軸は一羽ずつの体重を種類別に平均した値です。棒の高さを比べた後，それぞれ何件から計算した平均かを表で確かめます。

In [ ]:
lecture_masses = measured.groupby("species")["body_mass_g"]
lecture_mean_mass = lecture_masses.mean()
lecture_mass_summary = lecture_masses.size().to_frame("records")
lecture_mass_summary["mean_body_mass_g"] = lecture_mean_mass
lecture_mass_summary.round(1)

In [ ]:
fig, ax = plt.subplots(layout="constrained")
ax.bar(lecture_mean_mass.index, lecture_mean_mass, color="#586c7a")
ax.set(xlabel="Species", ylabel="Mean body mass (g)", ylim=(0, 5500))
plt.show()

### 3. 一羽ずつの体重の分布を比べる

平均が近い Adelie と Chinstrap では，個体の体重も同じ範囲に多いのでしょうか。三つのヒストグラムを同じ区間幅・同じ軸で描き，棒が重なる体重帯を探します。Gentoo の個体が多い体重帯も，同じ横位置で比べてください。

In [ ]:
# 提供した作図セル。三つの群を，同じ区間と軸で描きます。
lecture_adelie = measured.loc[measured["species"] == "Adelie"]
lecture_chinstrap = measured.loc[measured["species"] == "Chinstrap"]
lecture_gentoo = measured.loc[measured["species"] == "Gentoo"]

fig, (adelie_ax, chinstrap_ax, gentoo_ax) = plt.subplots(
    3, 1, sharex=True, sharey=True, figsize=(6.2, 5.8), layout="constrained",
)
mass_bins = range(2500, 6501, 250)
adelie_ax.hist(lecture_adelie["body_mass_g"], bins=mass_bins)
chinstrap_ax.hist(lecture_chinstrap["body_mass_g"], bins=mass_bins)
gentoo_ax.hist(lecture_gentoo["body_mass_g"], bins=mass_bins)
adelie_ax.set_title("Adelie (151 records)")
chinstrap_ax.set_title("Chinstrap (68 records)")
gentoo_ax.set(title="Gentoo (123 records)", xlabel="Body mass (g)",
              xlim=(2500, 6500), ylim=(0, 35))
fig.supylabel("Records")
plt.show()

### 4. 一羽の二つの値を一点にする

体重とは別の二つの測定値を，同じ一羽の組として見ます。横軸をくちばしの長さ，縦軸を厚みとし，行ラベル 0 の値を表から読み取ってください。その二値が交わる図の一点を，点線をたどって確かめます。

In [ ]:
lecture_bills = measured.loc[[0, 152, 276], ["species", "bill_length_mm", "bill_depth_mm"]]
lecture_bills

In [ ]:
fig, ax = plt.subplots(layout="constrained")
ax.scatter(lecture_bills["bill_length_mm"], lecture_bills["bill_depth_mm"], s=55)
ax.axvline(lecture_bills.loc[0, "bill_length_mm"], color="#586c7a", linestyle=":")
ax.axhline(lecture_bills.loc[0, "bill_depth_mm"], color="#586c7a", linestyle=":")
ax.set(xlabel="Bill length (mm)", ylabel="Bill depth (mm)", xlim=(30, 62), ylim=(12, 22))
plt.show()

### 5. 全記録を一枚に置く

同じ二つの列を 342 件すべてに広げます。長さが 46〜48 mm の付近には，厚みの違う点が並びます。灰色の点だけでは Chinstrap と Gentoo の違いを確かめられません。どの点がどのペンギンの種類かを示すには，元の表のどの列を使いますか。

In [ ]:
fig, ax = plt.subplots(layout="constrained")
ax.scatter(measured["bill_length_mm"], measured["bill_depth_mm"],
           color="#687c8a", s=22, alpha=0.6)
ax.set(xlabel="Bill length (mm)", ylabel="Bill depth (mm)",
       xlim=(30, 62), ylim=(12, 22))
plt.show()

### 6. ペンギンの種類を替えて繰り返す

初めは名前を表示するだけにします。`for` はリストから値を一つずつ `name` に入れ，字下げした行を実行します。表示される三つの名前と順番を予想してから実行してください。

In [ ]:
species_names = ["Adelie", "Chinstrap", "Gentoo"]

for name in species_names:
    print(name)

次は，`name` に入ったペンギンの種類の行を選びます。一回目の `group` は Adelie，二回目は Chinstrap の記録です。毎回，元の 342 件から条件に合う行を選び直します。出力の件数を見て，前の回の行を足していないことを確かめます。

In [ ]:
for name in species_names:
    group = measured.loc[measured["species"] == name]
    print(name, len(group))

### 7. 同じ場所へ描き足す

今度は `print()` の代わりに，その回の `group` のくちばしの長さと厚みを `scatter()` に渡します。`subplots()` を繰り返しの前に一度だけ実行するので，Adelie，Chinstrap，Gentoo の点が同じ軸に加わります。繰り返しの後で凡例を表示し，点の色とペンギンの種類を対応させます。

灰色の図と点の位置を照合し，長さが近い Chinstrap と Gentoo の点では，どちらが厚い側に多いか読んでください。

In [ ]:
colors = {"Adelie": "#286b70", "Chinstrap": "#b15334", "Gentoo": "#695498"}
markers = {"Adelie": "o", "Chinstrap": "^", "Gentoo": "s"}

fig, ax = plt.subplots(layout="constrained")
for name in species_names:
    group = measured.loc[measured["species"] == name]
    ax.scatter(group["bill_length_mm"], group["bill_depth_mm"], label=name,
               color=colors[name], marker=markers[name], s=24, alpha=0.65)
ax.set(xlabel="Bill length (mm)", ylabel="Bill depth (mm)",
       xlim=(30, 62), ylim=(12, 22))
ax.legend()
plt.show()

## 自分で分析する実習（約20分）

### 1. 依頼と図を選ぶ（5分）

本文では，平均体重が近い Adelie と Chinstrap でも，別の測定値を比べると違いが見えました。今度は翼の長さを使い，次のどちらか一つの問いに答えます。

- **A：翼の長さの分布。** Adelie，Chinstrap，Gentoo それぞれで，個体の翼がどの長さに集まり，ほかとどこで重なるかを伝える。
- **B：翼の長さと体重。** 翼が長い個体は体重も重いかを，ペンギンの種類による違いと，Adelie の中での広がりなどから伝える。

両方を行う必要はありません。まず本文の平均体重の棒だけで選んだ依頼に答えられるか考え，ヒストグラムか散布図を一つ選びます。`flipper_length_mm` は翼の長さ（mm），`body_mass_g` は体重（g）です。先に下の表で，一羽に対応する値を読みます。

In [ ]:
measured[["species", "flipper_length_mm", "body_mass_g"]].head(4)

**図を見る前に記入します。**

- 選んだ依頼：
- 使う図と必要な列：
- 棒または点が何を表すか，平均の棒だけでは足りない理由：
- ペンギンの種類別に見ると分かりそうなことの予想：

### 2. 問いに合う図を作る（9分）

次の共通設定を実行し，その後は**選んだ図の骨組みだけ**を記入します。もう一方はコメントのまま残せます。Adelie，Chinstrap，Gentoo の計 342 件を使い，軸の範囲をそろえます。

In [ ]:
species_names = ["Adelie", "Chinstrap", "Gentoo"]
colors = {"Adelie": "#286b70", "Chinstrap": "#b15334", "Gentoo": "#695498"}
markers = {"Adelie": "o", "Chinstrap": "^", "Gentoo": "s"}

#### ヒストグラムを選んだ場合

翼の長さ一列を区間に分け，そこに入る個体の記録を数えます。ペンギンの種類ごとに一枚ずつ描くため，`subplots()` を `for` の内側に置きます。三枚を同じ軸で比べれば，A の問いにある重なりを読めます。

下の行の先頭の `#` と直後の空白1個を外し，`group[""]` に使う列名，`xlabel` と `ylabel` に軸名を記入してください。`for` の内側の字下げは残します。区間は5 mm幅，表示範囲は全記録が入る170〜235 mmで共通です。

In [ ]:
# for name in species_names:
#     group = measured.loc[measured["species"] == name]
#     fig, ax = plt.subplots(layout="constrained")
#     ax.hist(group[""], bins=range(170, 236, 5), color=colors[name], edgecolor="white")
#     ax.set(title=name, xlabel="", ylabel="", xlim=(170, 235), ylim=(0, 50))
#     plt.show()

#### 散布図を選んだ場合

同じ一羽の翼の長さと体重を，一つの点にします。三つのペンギンの種類を同じ軸に重ねるため，`subplots()` は `for` の前に置きます。B の問いでは，全体の点の配置と，Adelie など同じ名前の個体に限った広がりの両方を見ます。

下の行の先頭の `#` と直後の空白1個を外し，二つの `group[""]` に列名，`xlabel` と `ylabel` に単位を含む軸名を記入します。`for` の内側の字下げは残します。比較しやすいよう，横を翼の長さ，縦を体重とする骨組みです。

In [ ]:
# fig, ax = plt.subplots(layout="constrained")
# for name in species_names:
#     group = measured.loc[measured["species"] == name]
#     ax.scatter(group[""], group[""], label=name,
#                color=colors[name], marker=markers[name], s=30, alpha=0.65)
# ax.set(xlabel="", ylabel="", xlim=(170, 235), ylim=(2500, 6500))
# ax.legend(loc="upper left")
# plt.show()

<details>
<summary>列と軸の対応を確認する</summary>

翼の長さを区間に分けるなら，`hist()` に `group["flipper_length_mm"]` を渡します。横軸名は `Flipper length (mm)`，縦軸名は件数を表す `Records` です。

一羽の翼と体重の対応を描くなら，`scatter()` に `group["flipper_length_mm"]` と `group["body_mass_g"]` をこの順で渡します。横軸名は `Flipper length (mm)`，縦軸名は `Body mass (g)` です。

図の種類を選び直した場合は，最初の理由も結果を踏まえて書き直して構いません。変えた理由を一言残します。
</details>

### 3. 図で答えられることを伝える（6分）

次の三点を短い段落にまとめます。図の見た目だけでなく，依頼への答えを書いてください。

1. **予想と照らした発見。** A なら多く集まる区間と重なる区間を，B なら Adelie など一つのペンギンの種類の中での翼と体重の関係を，名前と軸上の値を一例以上挙げて説明する。予想が変わった場合は，どの出力で変わったかも書く。
2. **この図に残らない情報。** 選ばなかった依頼へも答えるには，どの列や対応を図に加えるか，または何を数え直す必要があるか。自分の図で「翼が200 mm付近」の記録を例に考える。
3. **対象。** 調査時期と図に使った件数を添える。ペンギンの種類別の棒の高さを比べる場合は，元の件数が 151，68，123 と異なることにも触れる。

**報告：**

（ここに記入）

提出前にカーネルを再起動し，全セルを上から実行します。選んだ図と，依頼・選択理由・報告が対応したノートブックを保存してください。選ばなかった図のコードはコメントのまま残します。

#### 出典

2007〜2009年に南極半島付近の3島で調査された成鳥の記録です。Kristen Gorman と Palmer Station LTER の研究者による調査から，Horst，Hill，Gorman が整理・配布した表を使用しています。[データの配布元](https://allisonhorst.github.io/palmerpenguins/) ・ [原著](https://doi.org/10.1371/journal.pone.0090081)